# 08 · Strongly nonlinear 2D reference and observations

**Run All**. A converged conservative numerical reference and two frozen observation sets are already included; this notebook verifies and reuses them. The user’s original figure files and the earlier 39 trainings are untouched.

We solve the quasilinear **Fourier heat equation in conservative form**

$$\partial_t\theta=\mathrm{Fo}\,\nabla\cdot[a(\theta)\nabla\theta],\quad \mathrm{Fo}=0.002,\quad (x,y)\in(0,1)^2,\quad T=1,$$

with $\theta=0$ on all four boundaries and no source. The coefficient is smooth, positive, and temperature dependent:

$$a(\theta)=1+\frac{19}{1+\exp[-12(\theta-0.5)]}.$$

It varies by approximately 20-fold across the initial field. The initial temperature is a **nontrivial, positive, two-hotspot field**, not a pre-specified two-eigenmode family. Only the *terminal interior samples* enter the inverse objective; initial truth is retained for later evaluation.

The forward solver uses face-centred conservative fluxes and Picard-converged backward Euler, **independent of the neural model**. This is a synthetic benchmark with a deliberately strong constitutive dependence, not a measured material-law claim.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

ROOT=Path.cwd().resolve()
if not (ROOT/'inverse_study').is_dir() and (ROOT/'heat'/'inverse_study').is_dir(): ROOT=ROOT/'heat'
if not (ROOT/'inverse_study').is_dir(): raise FileNotFoundError('Open this notebook inside the heat project folder')
sys.path.insert(0,str(ROOT))
from inverse_study import nonlinear2d as n2, nonlinear2d_figures as plot2
R=ROOT/'results'/'nonlinear2d'
R.mkdir(parents=True,exist_ok=True)
p=n2.problem_spec()
print('PDE law:',p['law'],'Fo=',p['Fo'],'T=',p['T'])
print('Diffusivity range at 0<=theta<=1:',n2.coefficient([0,1],p))

## Cached independent references and grid refinement

Three **real numerical** forward calculations are included, rather than placeholders. The initial profile is normalized with a fixed mesh-independent constant. The solver refuses a cached reference if the underlying problem definition changes. No model training is performed here.

In [ ]:
settings=[(33,0.02),(65,0.01),(97,0.005)]
refs={}
for size,dt in settings:
    path=R/'reference'/f'reference_n{size}_dt{dt:g}.npz'
    refs[size],status=n2.ensure_reference(path,size,dt)
    print(f'{size}x{size}, Δt={dt:.4g}: {status}, max Picard iterations={refs[size]["max_picard_iterations"]}')
for size in [33,65]:
    row=n2.compare_reference(refs[size],refs[97],n_evaluation=33)
    print(f'{size} vs 97, terminal relative grid/time difference: {row["terminal_relative_l2"]:.6f}')
assert n2.compare_reference(refs[65],refs[97],33)['terminal_relative_l2'] < 0.005
ref=refs[97]
print('Initial coefficient min,max:', ref['diffusivity_range_initial'])
print('Terminal coefficient min,max:', ref['diffusivity_range_terminal'])

## Linear-limit verification and nonlinearity diagnostic

The same conservative solver must approach the analytical Dirichlet eigenmode solution when $a\equiv1$. We also calculate the terminal discrepancy of a cold-coefficient ($a=1$) problem with the **same two-hotspot initial condition**. This is a diagnostic of nonlinearity, **not a classical inverse baseline**.

In [ ]:
# A small fresh linear-mode check, independent of the nonlinear reference.
xx_axis,xg,yg=n2.grid(65)
phi=np.sin(np.pi*xg)*np.sin(np.pi*yg)
linear_spec=p|{'a_lo':1.0,'a_hi':1.0}
linear_mode=n2.forward(65,0.01,spec=linear_spec,initial_field=phi)
expected=phi*np.exp(-2*np.pi**2*p['Fo']*p['T'])
error=n2.relative_l2(linear_mode['terminal'],expected)
print('Exact linear eigenmode relative forward error:',error)
assert error<5e-4
cold=n2.forward(65,0.01,spec=linear_spec)
nonlin_difference=n2.relative_l2(cold['terminal'],refs[65]['terminal'])
print('Terminal nonlinear-vs-a=1 relative difference:',nonlin_difference)
assert nonlin_difference>0.15

## Freeze the matched terminal observations

Two datasets are formed at $31\times31=961$ **interior** sensors from the same 97²-node forward reference: clean and 5% RMS-scaled iid Gaussian measurement noise. All neural methods will read the **identical frozen vector**, and boundary data remain exact zero. The truth is kept in the reference file, not in training.

In [ ]:
for level,seed in [(0.0,0),(0.05,3100)]:
    folder,obs=n2.create_observations(ref,noise=level,seed=seed,n_obs_grid=33,root=R)
    print(f'noise={level:g}, seed={seed}, sensors={len(obs["y_obs"])}, ||noise||₂={obs["noise_norm"]:.6g}; {folder.relative_to(ROOT)}')

## Reference visualization in the **author's original** plot style

The `jet` palette, 16-level translucent contours, equal axes, scientific labels and panel letters follow `epinn_publication_figures_selected_only(2).ipynb` from the supplied research archive. The original PNG figures are not changed. The image below contains **only calculated forward reference fields**, no invented inverse reconstruction.

In [ ]:
base=R/'figures'/'reference_original_style'
plot2.plot_reference(ref,base)
if (base.with_suffix('.png')).exists(): display(Image(filename=str(base.with_suffix('.png'))))
print('Ready for 09_Strongly_nonlinear_2D_inverse_production.ipynb')